<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/06_Quantificadores_e_Predicados_em_Redes_de_Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/marinazakimi/ECAA08--Grupo-06/blob/main/etapa-01-logica/06_Quantificadores_e_Predicados_em_Redes_de_Sensores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 06 - Quantificadores e Predicados em Redes de Sensores

## Sistema SCADA aplicado a um Drone Agrícola de Pulverização — Grupo 06

Este notebook implementa a avaliação de **predicados** e dos **quantificadores lógicos** $\forall$ (*FORALL*) e $\exists$ (*EXISTS*) sobre coleções de sensores distribuídos na planta — tanto na barra de pulverização setorizada do drone quanto na rede completa de telemetria (estação de solo + drone).

Serão implementados:

- Operadores genéricos `FORALL` e `EXISTS` sobre domínios finitos;
- Um modelo de **barra de pulverização setorizada** (múltiplos bicos/seções, cada um com seu próprio sensor de pressão e vazão);
- Varredura global de **falha de comunicação** na rede de sensores (estação + drone);
- Verificação computacional das **equivalências de De Morgan quantificadas**;
- Uma função de varredura de estado que pode alimentar o motor de diagnóstico da Aula 09.


---
## 1. Fundamentação Teórica: Lógica de Predicados e Quantificadores

Enquanto a lógica proposicional (Aulas 03–05) trata sentenças atômicas indivisíveis, a **Lógica de Predicados** permite parametrizar propriedades sobre um domínio finito de ativos industriais — no nosso caso, os setores da barra de pulverização e os instrumentos da rede de telemetria.

1. **Predicado $P(x)$:** função booleana $P: U \rightarrow \{0, 1\}$, onde $U$ é o universo de discurso (ex: conjunto de setores de pulverização $\mathcal{S}$, conjunto de sensores da rede $\mathcal{R}$).
2. **Quantificador Universal ($\forall x \in U,\; P(x)$):** "para todo $x$ em $U$, $P(x)$ é Verdadeiro". Em domínio finito $U = \{x_1, \dots, x_n\}$:
   $$\forall x\, P(x) \equiv P(x_1) \land P(x_2) \land \dots \land P(x_n)$$
3. **Quantificador Existencial ($\exists x \in U,\; P(x)$):** "existe ao menos um $x$ em $U$ tal que $P(x)$ é Verdadeiro". Em domínio finito:
   $$\exists x\, P(x) \equiv P(x_1) \lor P(x_2) \lor \dots \lor P(x_n)$$
4. **Equivalências de De Morgan Quantificadas:**
   $$\neg (\forall x\, P(x)) \equiv \exists x\, \neg P(x)$$
   $$\neg (\exists x\, P(x)) \equiv \forall x\, \neg P(x)$$

No projeto SCADA-Core, esses operadores permitem escrever regras de varredura **independentes do número de sensores** — se a barra de pulverização tiver 4 ou 12 seções, a mesma expressão $\exists s \in \mathcal{S}, \; \text{Obstruido}(s)$ continua válida sem reescrever a lógica.


In [ ]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)


from dataclasses import dataclass
from typing import List, Callable, Any


def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return all(predicado(x) for x in dominio)


def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return any(predicado(x) for x in dominio)


print("Operadores de quantificação FORALL / EXISTS carregados com sucesso.")


Operadores de quantificação FORALL / EXISTS carregados com sucesso.


---
## 2. Domínio 1: Barra de Pulverização Setorizada

De acordo com o mapeamento de insumos do projeto, as válvulas seccionadoras (`DRN_XV_01`) permitem controlar quais setores dos bicos permanecem ativos durante a pulverização. Cada seção da barra recebe seu próprio sensor de pressão e vazão, formando uma pequena rede de instrumentos distribuídos sobre o mesmo atuador de pulverização.

Modelamos cada setor como um objeto `SetorPulverizacao`, análogo ao vetor de proposições atômicas definido na Aula 02, porém agora replicado por seção.


In [ ]:
@dataclass
class SetorPulverizacao:
    tag: str
    pressao: float          # bar (DRN_PT_01x)
    vazao: float             # L/min (DRN_FT_01x)
    bomba_ligada: bool
    falha_comunicacao: bool = False


# Limiares consistentes com a Aula 02 (P_PRESS_HIGH e P_FLOW_LOW)
LIMITE_PRESSAO_ALTA = 4.5   # bar
LIMITE_VAZAO_BAIXA = 0.4    # L/min

barra_pulverizacao = [
    SetorPulverizacao(tag="DRN_PT_01A / DRN_FT_01A", pressao=3.0, vazao=2.1, bomba_ligada=True),
    SetorPulverizacao(tag="DRN_PT_01B / DRN_FT_01B", pressao=3.2, vazao=1.9, bomba_ligada=True),
    SetorPulverizacao(tag="DRN_PT_01C / DRN_FT_01C", pressao=5.2, vazao=0.25, bomba_ligada=True),  # obstruído
    SetorPulverizacao(tag="DRN_PT_01D / DRN_FT_01D", pressao=3.1, vazao=2.0, bomba_ligada=True),
]


def obstruido(setor: SetorPulverizacao) -> bool:
    """Mesmo padrão da regra R3 (Aula 09), porém aplicado por setor individual."""
    return setor.bomba_ligada and setor.pressao >= LIMITE_PRESSAO_ALTA and setor.vazao <= LIMITE_VAZAO_BAIXA


existe_setor_obstruido = EXISTS(barra_pulverizacao, obstruido)
todos_setores_ok = FORALL(barra_pulverizacao, lambda s: not obstruido(s))

print(f"1. Existe setor obstruído (EXISTS): {existe_setor_obstruido}")
print(f"2. Todos os setores operando normalmente (FORALL): {todos_setores_ok}")

tabela_setores = [
    {
        "Setor": s.tag,
        "Pressão (bar)": s.pressao,
        "Vazão (L/min)": s.vazao,
        "Obstruído?": obstruido(s),
    }
    for s in barra_pulverizacao
]
print("\n" + formatar_tabela(tabela_setores))

assert existe_setor_obstruido is True
assert todos_setores_ok is False


1. Existe setor obstruído (EXISTS): True
2. Todos os setores operando normalmente (FORALL): False

Setor                   | Pressão (bar) | Vazão (L/min) | Obstruído?
------------------------+---------------+---------------+-----------
DRN_PT_01A / DRN_FT_01A | 3.0           | 2.1           | False     
DRN_PT_01B / DRN_FT_01B | 3.2           | 1.9           | False     
DRN_PT_01C / DRN_FT_01C | 5.2           | 0.25          | True      
DRN_PT_01D / DRN_FT_01D | 3.1           | 2.0           | False     


Observe que, com apenas **um** dos quatro setores em condição de obstrução, o predicado $\exists s, \text{Obstruido}(s)$ já é suficiente para acionar o alarme, enquanto $\forall s, \neg\text{Obstruido}(s)$ corretamente falha — nenhuma seção isolada precisa ser verificada manualmente pelo operador.

---
## 3. Domínio 2: Rede Completa de Telemetria (Estação + Drone)

O segundo domínio de varredura cobre **toda a rede de sensores** do sistema SCADA, reunindo os instrumentos da estação de solo e do drone catalogados na Aula 02. Aqui o predicado de interesse não é a leitura da grandeza física, mas a **integridade da comunicação** de cada instrumento — um pré-requisito para que a Aula 04 possa avaliar os permissivos de decolagem com segurança.


In [ ]:
@dataclass
class SensorRede:
    tag: str
    local: str
    falha_comunicacao: bool = False


rede_sensores = [
    SensorRede(tag="EST_LT_01", local="Estação de Solo"),
    SensorRede(tag="EST_WT_01", local="Estação de Solo"),
    SensorRede(tag="EST_TT_01", local="Estação de Solo"),
    SensorRede(tag="DRN_LT_01", local="Drone"),
    SensorRede(tag="DRN_PT_01", local="Drone"),
    SensorRede(tag="DRN_FT_01", local="Drone"),
    SensorRede(tag="DRN_ET_01", local="Drone"),
    SensorRede(tag="DRN_ZT_01", local="Drone"),
    SensorRede(tag="DRN_GPS_01", local="Drone", falha_comunicacao=True),  # telemetria intermitente
]

todos_comunicando = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)
existe_falha_comunicacao = EXISTS(rede_sensores, lambda s: s.falha_comunicacao)

print(f"3. Todos os sensores da rede comunicando (FORALL): {todos_comunicando}")
print(f"4. Existe ao menos um sensor com falha de comunicação (EXISTS): {existe_falha_comunicacao}")

tabela_rede = [
    {"Tag": s.tag, "Local": s.local, "Falha de Comunicação": s.falha_comunicacao}
    for s in rede_sensores
]
print("\n" + formatar_tabela(tabela_rede))

assert todos_comunicando is False
assert existe_falha_comunicacao is True


3. Todos os sensores da rede comunicando (FORALL): False
4. Existe ao menos um sensor com falha de comunicação (EXISTS): True

Tag        | Local           | Falha de Comunicação
-----------+-----------------+---------------------
EST_LT_01  | Estação de Solo | False               
EST_WT_01  | Estação de Solo | False               
EST_TT_01  | Estação de Solo | False               
DRN_LT_01  | Drone           | False               
DRN_PT_01  | Drone           | False               
DRN_FT_01  | Drone           | False               
DRN_ET_01  | Drone           | False               
DRN_ZT_01  | Drone           | False               
DRN_GPS_01 | Drone           | True                


Neste cenário, a falha simulada no `DRN_GPS_01` já é suficiente para reprovar o predicado $\forall s, \neg\text{FalhaComm}(s)$. Como a Aula 04 exige `gps_ok = True` no permissivo de decolagem, essa varredura de rede funciona como um **pré-requisito lógico** anterior à própria avaliação do permissivo: se a rede não está integralmente comunicando, nem faz sentido avaliar os permissivos com dados potencialmente obsoletos.

---
## 4. Verificação Computacional das Equivalências de De Morgan Quantificadas

A teoria afirma que $\neg(\forall x\, P(x)) \equiv \exists x\, \neg P(x)$. Verificamos essa equivalência computacionalmente sobre os dois domínios definidos acima, comparando o resultado de negar o `FORALL` com o resultado de aplicar `EXISTS` ao predicado negado.


In [ ]:
def nao(predicado):
    return lambda x: not predicado(x)


# Verificação sobre a barra de pulverização
lado_esquerdo_1 = not FORALL(barra_pulverizacao, lambda s: not obstruido(s))
lado_direito_1 = EXISTS(barra_pulverizacao, obstruido)
print(f"¬(∀s, ¬Obstruido(s)) = {lado_esquerdo_1}   |   ∃s, Obstruido(s) = {lado_direito_1}   ->  Equivalentes: {lado_esquerdo_1 == lado_direito_1}")

# Verificação sobre a rede de sensores
lado_esquerdo_2 = not FORALL(rede_sensores, nao(lambda s: s.falha_comunicacao))
lado_direito_2 = EXISTS(rede_sensores, lambda s: s.falha_comunicacao)
print(f"¬(∀s, ¬FalhaComm(s)) = {lado_esquerdo_2}   |   ∃s, FalhaComm(s) = {lado_direito_2}   ->  Equivalentes: {lado_esquerdo_2 == lado_direito_2}")

assert lado_esquerdo_1 == lado_direito_1
assert lado_esquerdo_2 == lado_direito_2
print("\nEquivalência de De Morgan quantificada confirmada nos dois domínios.")


¬(∀s, ¬Obstruido(s)) = True   |   ∃s, Obstruido(s) = True   ->  Equivalentes: True
¬(∀s, ¬FalhaComm(s)) = True   |   ∃s, FalhaComm(s) = True   ->  Equivalentes: True

Equivalência de De Morgan quantificada confirmada nos dois domínios.


---
## 5. Motor de Varredura de Estado

Reunindo os dois domínios em uma única função, obtemos o **módulo de varredura global** entregável desta aula. Ele resume, a partir dos dois domínios (setores de pulverização e rede de telemetria), se a planta está apta a operar sob o ponto de vista de predicados quantificados — resultado que pode ser consumido tanto pelo bloco de permissivos (Aula 04) quanto pelo motor de diagnóstico (Aula 09), generalizando a regra R3 (`bomba_ligada AND vazao_baixa AND pressao_alta`) para qualquer número de setores.


In [ ]:
def motor_varredura_estado(setores: List[SetorPulverizacao], rede: List[SensorRede]) -> dict:
    existe_obstrucao = EXISTS(setores, obstruido)
    barra_integra = FORALL(setores, lambda s: not obstruido(s))

    rede_integra = FORALL(rede, lambda s: not s.falha_comunicacao)
    existe_falha_rede = EXISTS(rede, lambda s: s.falha_comunicacao)

    return {
        "existe_setor_obstruido": existe_obstrucao,
        "barra_pulverizacao_integra": barra_integra,
        "rede_telemetria_integra": rede_integra,
        "existe_falha_comunicacao": existe_falha_rede,
        "varredura_permite_operacao": barra_integra and rede_integra,
    }


resultado_varredura = motor_varredura_estado(barra_pulverizacao, rede_sensores)

print("=== MOTOR DE VARREDURA DE ESTADO (Aula 06) ===\n")
for chave, valor in resultado_varredura.items():
    print(f"{chave}: {valor}")


=== MOTOR DE VARREDURA DE ESTADO (Aula 06) ===

existe_setor_obstruido: True
barra_pulverizacao_integra: False
rede_telemetria_integra: False
existe_falha_comunicacao: True
varredura_permite_operacao: False


---
## 6. Aplicação Prática em Controle e Automação

- **Escalabilidade da lógica de alarmes:** a regra R3 do motor de diagnóstico (Aula 09) foi originalmente escrita para uma única bomba/linha. Com o predicado `obstruido(s)` e o operador `EXISTS`, a mesma lógica de obstrução passa a valer para qualquer quantidade de setores na barra, sem reescrever a base de regras a cada nova seção instalada.
- **Pré-requisito de integridade de rede:** o predicado $\forall s, \neg\text{FalhaComm}(s)$ funciona como uma verificação anterior aos permissivos de decolagem e pulverização (Aula 04) — não há sentido em avaliar `gps_ok` ou `bat_low` se o canal de telemetria correspondente está intermitente.
- **Diagnóstico setorial mais preciso:** ao identificar *qual* setor específico está obstruído (e não apenas que "existe uma obstrução"), a HMI pode direcionar o operador diretamente ao bico com problema, reduzindo o tempo de manutenção em campo.

---
## 7. Mapeamento para Provas e Exames Tecnológicos

- **Equivalências de De Morgan Quantificadas:** $\neg(\forall x P(x)) \equiv \exists x \neg P(x)$ e $\neg(\exists x P(x)) \equiv \forall x \neg P(x)$, verificadas computacionalmente na Seção 4.
- **Expansão de quantificadores em domínio finito:** todo $\forall$/$\exists$ sobre um conjunto finito de sensores é redutível a uma cadeia de conjunções/disjunções proposicionais (Aulas 03–05), o que permite reaproveitar toda a álgebra booleana já demonstrada nessas aulas.

---
## 8. Análise de Erros Conceituais e Numéricos

- **Quantificador Universal sobre Domínio Vazio (Vacuous Truth):** se a lista `barra_pulverizacao` estiver vazia (nenhum setor cadastrado), `FORALL` retorna `True` por definição matemática — o sistema reportaria "todos os setores OK" sem ter verificado sensor algum. É essencial garantir, antes da varredura, que o domínio nunca esteja vazio em operação real.
- **Troca indevida de $\forall$ por $\exists$ no permissivo geral:** usar `EXISTS(setores, lambda s: not obstruido(s))` no lugar de `FORALL` liberaria a pulverização assim que **um único** setor estivesse saudável, ignorando obstruções nos demais — um erro de modelagem que mascararia falhas reais.
- **Confundir "falha de comunicação" com "leitura fora da faixa":** um sensor com `falha_comunicacao=True` não fornece leitura confiável alguma; tratá-lo como se estivesse reportando um valor normal (ex: assumir pressão = 0 na ausência de dado) pode mascarar tanto uma obstrução quanto uma ruptura de mangueira.


---
## Conclusão

Este notebook estendeu a lógica proposicional das Aulas 03–05 para a **Lógica de Predicados**, introduzindo os quantificadores $\forall$ e $\exists$ como mecanismo de varredura sobre coleções de sensores — tanto na barra de pulverização setorizada quanto na rede completa de telemetria do drone e da estação de solo.

A principal contribuição prática é a **generalização** das regras de alarme e permissivo: em vez de reescrever expressões booleanas para cada novo sensor ou setor instalado na planta, o sistema SCADA passa a expressar suas regras em função de um domínio de instrumentos, tornando a lógica de segurança escalável. O `motor_varredura_estado` implementado aqui complementa o motor de intertravamento (Aula 04) e alimenta a generalização da regra de obstrução do motor de diagnóstico (Aula 09), sendo consolidado com os demais motores na avaliação do Módulo 1 (Aula 10).
